In [10]:
from pathlib import Path
import os
import json
import hashlib
import pandas as pd
import anthropic

from dotenv import load_dotenv
from neo4j import GraphDatabase

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")

CSV_PATH = ROOT / "data" / "processed" / "property_listings_raw.csv"
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5")

assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"
assert os.getenv("NEO4J_URI"), "NEO4J_URI missing from .env"
assert os.getenv("NEO4J_PASSWORD"), "NEO4J_PASSWORD missing from .env"
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing from .env"

df = pd.read_csv(CSV_PATH, low_memory=False)

neo4j_driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(
        os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j")),
        os.environ["NEO4J_PASSWORD"],
    ),
)
neo4j_driver.verify_connectivity()

llm_client = anthropic.Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"]
)

print("CSV rows:", len(df))
print("CSV columns:", df.columns.tolist())
print("Connected to Neo4j and Anthropic.")


CSV rows: 6495
CSV columns: ['id', 'crm_id', 'strapi_id', 'slug', 'title', 'listing_type', 'department', 'search_type', 'status', 'price', 'price_qualifier', 'beds', 'bathrooms', 'area', 'community', 'building', 'display_address', 'address.address1', 'address.address2', 'address.address3', 'address.address4', 'latitude', 'longitude', 'floorarea_min', 'floorarea_type', 'description', 'long_description', 'accommodation_summary', 'new_development', 'link', 'maps_url', 'source', 'images', 'image_count', 'scraped_at', 'introtext', 'special', 'brochure_url', 'dld_listing_number', 'dld_permit_type', 'dld_qr_url', 'updated_at', 'completion_status', 'cheques', 'plot_area_min', 'agent_name', 'agent_email', 'agent_phone', 'agent_whatsapp', 'agent_designation', 'agent_languages', 'agent_photo_url', 'agent_slug', 'video_tour', 'virtual_tour', 'enriched_at', 'property_type']
Connected to Neo4j and Anthropic.


In [15]:
def run_cypher(query, parameters=None):
    with neo4j_driver.session(database=NEO4J_DATABASE) as session:
        return [
            record.data()
            for record in session.run(query, parameters or {})
        ]


def load_intent_taxonomy():
    rows = run_cypher("""
        MATCH (i:IntentCategory)
        RETURN i.name AS name,
               i.description AS description
        ORDER BY i.name
    """)

    if not rows:
        raise RuntimeError(
            "No IntentCategory nodes found."
        )

    return rows


intent_taxonomy = load_intent_taxonomy()

print("Configured intent categories:")
for item in intent_taxonomy:
    print(f"- {item['name']}: {item['description']}")

Configured intent categories:
- Family-Friendly: Verified evidence relevant to family living
- Investment: Verified investment evidence or market metrics
- Lifestyle: Verified lifestyle amenities and surroundings
- Views: Explicitly verified property views


In [17]:
INTENT_TOOL = {
    "name": "submit_intent_candidates",
    "description": (
        "Return intent categories supported directly by the supplied "
        "property text. Quote evidence verbatim. Do not infer missing "
        "amenities, views, commute times, or neighbourhood facts."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "intents": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "category": {"type": "string"},
                        "evidence": {"type": "string"},
                        "source_field": {"type": "string"},
                    },
                    "required": [
                        "category",
                        "evidence",
                        "source_field",
                    ],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["intents"],
        "additionalProperties": False,
    },
}


def extract_intents(property_record, taxonomy=None):
    taxonomy = taxonomy if taxonomy is not None else intent_taxonomy

    allowed_categories = {
        item["name"] for item in taxonomy
    }

    # Prepare source text without turning missing values into "nan".
    sources = {}

    for field in ("description", "long_description"):
        value = property_record.get(field, "")

        if pd.isna(value):
            continue

        text = str(value).strip()

        if text and text.lower() != "nan":
            sources[field] = text

    if not sources:
        return []

    response = llm_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1500,
        system=(
            '''
            STRICT EVIDENCE RULES:
            - Family-Friendly: Require explicit family suitability evidence,
            such as family-oriented facilities or an explicitly family-friendly
            environment. Bedroom count alone is not sufficient.
            - Views: Require an explicit view, outlook, or panorama described
            in the source text. A balcony alone is not sufficient.
            - Lifestyle: Require explicit lifestyle amenities or features,
            such as a pool, gym, garden, beach access, or walking paths.
            - Investment: Require explicit investment-related evidence, such as
            rental yield, ROI, rental demand, or capital appreciation claims.
            Do not infer investment potential from price alone.
            - Never infer commute times, distances, or commute convenience.
            - Use only evidence present in the property's description fields.
            - Return no intent when the evidence is insufficient.
            - Evidence must be copied exactly from its stated source field.
            '''
        ),
        tools=[INTENT_TOOL],
        tool_choice={
            "type": "tool",
            "name": "submit_intent_candidates",
        },
        messages=[
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "allowed_categories": taxonomy,
                        "property_id": str(property_record.get("id", "")),
                        "source_fields": sources,
                    },
                    ensure_ascii=False,
                ),
            }
        ],
    )

    tool_result = next(
        (
            block.input
            for block in response.content
            if block.type == "tool_use"
            and block.name == "submit_intent_candidates"
        ),
        None,
    )

    if tool_result is None:
        raise ValueError(
            "The model returned no structured intent result."
        )

    validated = []

    for candidate in tool_result["intents"]:
        category = candidate["category"]
        evidence = candidate["evidence"]
        source_field = candidate["source_field"]

        # Reject unknown categories.
        if category not in allowed_categories:
            continue

        # Reject invalid source fields or empty evidence.
        if source_field not in sources or not evidence.strip():
            continue

        # Require an exact quote from the named source field.
        if evidence not in sources[source_field]:
            continue

        validated.append(
            {
                "category": category,
                "evidence": evidence,
                "source_field": source_field,
                "extraction_method": "llm",
                "model": ANTHROPIC_MODEL,
            }
        )

    return validated

In [18]:
def extract_property_record(row):
    property_id = row.get("id")

    if pd.isna(property_id):
        return []

    record = row.to_dict()
    intents = extract_intents(record)

    return [
        {
            "property_id": str(property_id).strip(),
            **intent,
        }
        for intent in intents
    ]


sample_rows = df.head(10)
sample_results = []

for _, row in sample_rows.iterrows():
    sample_results.extend(extract_property_record(row))

sample_intents_df = pd.DataFrame(sample_results)

if sample_intents_df.empty:
    print("No supported intents were extracted from this sample.")
else:
    display(sample_intents_df)

print("Sample properties:", len(sample_rows))
print("Validated intent candidates:", len(sample_results))


,property_id,category,evidence,source_field,extraction_method,model
0,9971,Views,large balcony providing epic lake views,description,llm,claude-haiku-4-5-20251001
1,10051,Views,two bedroom apartment overlooking the pool,description,llm,claude-haiku-4-5-20251001
2,10051,Family-Friendly,The Greens is a popular Emaar community known ...,long_description,llm,claude-haiku-4-5-20251001
3,10051,Lifestyle,The Greens is a popular Emaar community known ...,long_description,llm,claude-haiku-4-5-20251001
4,6452,Lifestyle,within walking distance of the community pool ...,description,llm,claude-haiku-4-5-20251001
5,5191,Views,stunning view of Al Ittihad Park,description,llm,claude-haiku-4-5-20251001
6,5191,Lifestyle,"Residents benefit from beach, pool and gym access",description,llm,claude-haiku-4-5-20251001
7,19094,Lifestyle,A landscaped garden,description,llm,claude-haiku-4-5-20251001
8,20037,Lifestyle,landscaped garden,description,llm,claude-haiku-4-5-20251001
9,16266,Family-Friendly,fantastic family home in the sought after area...,long_description,llm,claude-haiku-4-5-20251001


Sample properties: 10
Validated intent candidates: 14


In [19]:
if sample_intents_df.empty:
    print("No intent candidates to review.")
else:
    display(
        sample_intents_df[
            ["property_id", "category", "evidence", "source_field"]
        ].sort_values(["property_id", "category"])
    )

    print("Properties reviewed:", len(sample_rows))
    print("Intent candidates:", len(sample_intents_df))
    print("Categories found:", sorted(sample_intents_df["category"].unique()))
    print(
        "Evidence duplicates:",
        int(sample_intents_df.duplicated(
            subset=["property_id", "category", "evidence"]
        ).sum())
    )

,property_id,category,evidence,source_field
2,10051,Family-Friendly,The Greens is a popular Emaar community known ...,long_description
3,10051,Lifestyle,The Greens is a popular Emaar community known ...,long_description
1,10051,Views,two bedroom apartment overlooking the pool,description
13,13947,Lifestyle,Located directly on the Marina Walk with immed...,long_description
12,13947,Views,Marina views,description
9,16266,Family-Friendly,fantastic family home in the sought after area...,long_description
10,16266,Lifestyle,within walking distance to a local park,long_description
7,19094,Lifestyle,A landscaped garden,description
8,20037,Lifestyle,landscaped garden,description
11,298,Views,two large balconies which provide peaceful com...,description


Properties reviewed: 10
Intent candidates: 14
Categories found: ['Family-Friendly', 'Lifestyle', 'Views']
Evidence duplicates: 0


In [11]:
# remove commute
commute_audit = run_cypher("""
MATCH (p:Property)-[r:HAS_INTENT]->(i:IntentCategory {name: "Commute"})
RETURN
    p.property_id AS property_id,
    r.source AS source,
    r.evidence AS evidence
ORDER BY p.property_id
LIMIT 30
""")

display(pd.DataFrame(commute_audit))
print("Relationships shown:", len(commute_audit))


""


Relationships shown: 0


In [14]:

commute_cleanup = run_cypher("""
MATCH (i:IntentCategory {name: "Commute"})
OPTIONAL MATCH ()-[r:HAS_INTENT]->(i)
WITH i, collect(r) AS relationships
FOREACH (rel IN relationships | DELETE rel)
WITH i
DELETE i
RETURN "Commute category removed" AS result
""")

print(commute_cleanup)


[{'result': 'Commute category removed'}]


In [16]:
intent_taxonomy = run_cypher("""
MATCH (i:IntentCategory)
RETURN i.name AS name, i.description AS description
ORDER BY i.name
""")

allowed_intents = {item["name"] for item in intent_taxonomy}

print("Active intent categories:", sorted(allowed_intents))
assert "Commute" not in allowed_intents, "Commute still exists in the loaded taxonomy."

Active intent categories: ['Family-Friendly', 'Investment', 'Lifestyle', 'Views']


In [20]:
required_columns = {"id", "description", "long_description"}
missing = required_columns - set(df.columns)

if missing:
    raise ValueError(f"Missing CSV columns: {sorted(missing)}")

print("Total CSV listings:", len(df))
print("Unique listing IDs:", df["id"].nunique())
print("Active intent categories:", sorted(allowed_intents))

Total CSV listings: 6495
Unique listing IDs: 6495
Active intent categories: ['Family-Friendly', 'Investment', 'Lifestyle', 'Views']


In [21]:
required = {"property_id", "category", "evidence", "source_field"}

if not sample_intents_df.empty:
    assert required.issubset(sample_intents_df.columns)

    invalid = sample_intents_df[
        sample_intents_df["evidence"].isna()
        | sample_intents_df["evidence"].astype(str).str.strip().eq("")
        | ~sample_intents_df["category"].isin(allowed_intents)
    ]

    print("Candidates:", len(sample_intents_df))
    print("Invalid candidates:", len(invalid))
    display(invalid)

Candidates: 14
Invalid candidates: 0


,property_id,category,evidence,source_field,extraction_method,model


## intent assigment

In [22]:
import hashlib

def listing_hash(row):
    source = {
        "description": row.get("description"),
        "long_description": row.get("long_description"),
    }

    normalized = {
        key: "" if pd.isna(value) else str(value).strip()
        for key, value in source.items()
    }

    return hashlib.sha256(
        json.dumps(normalized, sort_keys=True).encode("utf-8")
    ).hexdigest()

In [23]:
work_df = df.copy()

work_df["property_id"] = work_df["id"].map(
    lambda value: "" if pd.isna(value) else str(value).replace(".0", "").strip()
)
work_df["listing_hash"] = work_df.apply(listing_hash, axis=1)

work_df = work_df[work_df["property_id"] != ""].copy()

print("Listings ready:", len(work_df))
print("Unique property IDs:", work_df["property_id"].nunique())

Listings ready: 6495
Unique property IDs: 6495


In [24]:
BATCH_SIZE = 25

batch_results = []

for _, row in work_df.head(BATCH_SIZE).iterrows():
    intents = extract_property_record(row)

    for intent in intents:
        evidence = str(intent.get("evidence", "")).strip()

        if (
            intent.get("category") in allowed_intents
            and evidence
            and evidence in str(row.get(intent.get("source_field", ""), ""))
        ):
            batch_results.append({
                **intent,
                "listing_hash": row["listing_hash"],
            })

batch_intents_df = pd.DataFrame(batch_results)

print("Listings processed:", min(BATCH_SIZE, len(work_df)))
print("Validated intent records:", len(batch_intents_df))

display(batch_intents_df.head(20))

Listings processed: 25
Validated intent records: 42


,property_id,category,evidence,source_field,extraction_method,model,listing_hash
0,9971,Views,large balcony providing epic lake views,description,llm,claude-haiku-4-5-20251001,71c1fefd0055a0b6c8e9b3e004856e7684b160078752f2...
1,12292,Family-Friendly,3-bedroom Legacy villa,description,llm,claude-haiku-4-5-20251001,993fe088651b5fd2f72b48beedf6bd92f79ae72a7dfbc3...
2,10051,Views,overlooking the pool,description,llm,claude-haiku-4-5-20251001,9580d854597ff5ce11076be3f7614c3201793e6351cbd6...
3,10051,Family-Friendly,The Greens is a popular Emaar community known ...,long_description,llm,claude-haiku-4-5-20251001,9580d854597ff5ce11076be3f7614c3201793e6351cbd6...
4,6452,Lifestyle,within walking distance of the community pool ...,description,llm,claude-haiku-4-5-20251001,bb767584bae8662050decaf827d92421e5f1a214933b50...
5,6452,Lifestyle,within walking distance to the community pool ...,long_description,llm,claude-haiku-4-5-20251001,bb767584bae8662050decaf827d92421e5f1a214933b50...
6,5191,Views,stunning view of Al Ittihad Park,description,llm,claude-haiku-4-5-20251001,2149b3b98835aa359300e42586d16fd5ef21a756d91d3a...
7,5191,Lifestyle,"Residents benefit from beach, pool and gym access",description,llm,claude-haiku-4-5-20251001,2149b3b98835aa359300e42586d16fd5ef21a756d91d3a...
8,19094,Lifestyle,A landscaped garden,description,llm,claude-haiku-4-5-20251001,f1745ba001ba666e0e6dcd095fad45382c137dabdb9392...
9,20037,Lifestyle,landscaped garden,description,llm,claude-haiku-4-5-20251001,932b1851d312226f9d43f779d73ae537f25b49bf9548d8...
